In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# 1. Load and understand the dataset
df = pd.read_csv('insurance.csv')
print("Dataset Head:\n", df.head())
print("\nDataset Info:")
df.info()

# Brief dataset explanation string
dataset_desc = """
The dataset contains information about beneficiaries of a medical insurance plan.
Variables:
- age: Age of primary beneficiary
- sex: Insurance contractor gender, female, male
- bmi: Body mass index, providing an understanding of body, weights that are relatively high or low relative to height, objective index of body weight (kg / m ^ 2) using the ratio of height to weight, ideally 18.5 to 24.9
- children: Number of children covered by health insurance / Number of dependents
- smoker: Smoking
- region: the beneficiary's residential area in the US, northeast, southeast, southwest, northwest.
- charges: Individual medical costs billed by health insurance (Target Variable)
"""

# Visualization: Let's see the correlation and a scatter plot of bmi vs charges by smoker status
plt.figure(figsize=(10, 6))
sns.scatterplot(x='bmi', y='charges', hue='smoker', data=df)
plt.title('Medical Charges vs BMI (Colored by Smoker Status)')
plt.savefig('scatter.png')
plt.close()

# Preprocessing: Encoding categorical variables
df_encoded = pd.get_dummies(df, drop_first=True)

# 1. Identify features and target
X = df_encoded.drop('charges', axis=1)
y = df_encoded['charges']

# 2. Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4 & 5. Multiple Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(X_train_scaled, y_train)
y_pred_lin = lin_reg.predict(X_test_scaled)

# 6. Evaluate Linear Regression
print("\n--- Multiple Linear Regression Evaluation ---")
print(f"R-squared: {r2_score(y_test, y_pred_lin):.4f}")
print(f"MSE: {mean_squared_error(y_test, y_pred_lin):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred_lin):.4f}")

# 7. SVR with Hyperparameter Tuning
svr = SVR()
param_grid = {
    'kernel': ['linear', 'rbf'],
    'C': [100, 1000, 10000],
    'gamma': ['scale', 'auto']
}
grid_search = GridSearchCV(svr, param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

best_svr = grid_search.best_estimator_
y_pred_svr = best_svr.predict(X_test_scaled)

print("\n--- SVR Evaluation ---")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"R-squared: {r2_score(y_test, y_pred_svr):.4f}")
print(f"MSE: {mean_squared_error(y_test, y_pred_svr):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred_svr):.4f}")

Dataset Head:
    age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB

--- Multiple Linear Regression Evaluation ---
R-squared: 0.7836
M